# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarim-Ali-Shah/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring  
**Objective:** Construct a leak-free feature vector from historical performance windows, audit categorical handling and missing values, run adversarial leakage checks, and document all excluded fields.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We load `data/raw/content_refresh_anonymized.csv`, select valid retrospective lookback signals (impressions, clicks, CTR, position, update freshness, engagement), handle missing values deterministically, and engineer interaction features.


In [1]:
import pandas as pd
import numpy as np

# Load the raw anonymized starter catalog
data_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

# 1. Feature selection: strictly retrospective lookback features (90d)
feature_cols = [
    'impressions_90d',
    'clicks_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'content_age_days',
    'word_count',
    'engagement_rate',
    'scroll_rate'
]

# Create working copy
X = df[feature_cols].copy()

# 2. Missing value audit and imputation
print("Missing values before imputation:")
print(X.isnull().sum())

# Impute median for numeric features
for col in X.columns:
    if X[col].isnull().any():
        median_val = X[col].median()
        X[col] = X[col].fillna(median_val)

# 3. Feature Engineering: interaction and domain heuristics
# High exposure flag (>= 500 impressions)
X['is_high_exposure'] = (X['impressions_90d'] >= 500).astype(int)

# Content staleness flag (>= 180 days without update)
X['is_stale'] = (X['days_since_last_update'] >= 180).astype(int)

# Striking distance flag (position between 11 and 20)
X['is_striking_distance'] = ((X['avg_position'] >= 11) & (X['avg_position'] <= 20)).astype(int)

# Page 1 rank flag (position between 1 and 10)
X['is_page_one'] = (X['avg_position'] <= 10).astype(int)

# Clicks per day of age ratio (rate of return)
X['clicks_per_age_day'] = X['clicks_90d'] / np.maximum(X['content_age_days'], 1.0)

# Build target binary label for validation: decline > 15% forward trajectory
# In retrospective lookback, we define decline from trend_pct < -15%
y = (df['trend_pct'] < -15.0).astype(int)

print(f"\nFinal feature vector shape: {X.shape}")
print(f"Target label distribution: {y.value_counts(normalize=True).to_dict()}")
X.head()


Loaded dataset: 30,000 rows, 44 columns
Missing values before imputation:
impressions_90d              0
clicks_90d                   0
avg_position                 0
ctr                          0
days_since_last_update       0
content_age_days             0
word_count                7699
engagement_rate              0
scroll_rate                125
dtype: int64

Final feature vector shape: (30000, 14)
Target label distribution: {1: 0.578, 0: 0.422}


,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,engagement_rate,scroll_rate,is_high_exposure,is_stale,is_striking_distance,is_page_one,clicks_per_age_day
0,3803,29,10.6,0.76,20,187,3221.0,5.88,4.55,1,0,0,0,0.155080
1,15320,7,20.3,0.05,25,445,2481.0,0.00,10.00,1,0,0,0,0.015730
2,12581,11,36.5,0.09,20,141,3515.0,0.00,28.57,1,0,0,0,0.078014
3,11751,58,6.2,0.49,22,463,2877.0,1.28,3.45,1,0,0,1,0.125270
4,19140,24,44.0,0.13,14,263,2803.0,0.00,24.29,1,0,0,0,0.091255


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature Name | Type | Availability Moment | Missing Value Policy | Business Meaning |
|---|---|---|---|---|
| `impressions_90d` | Continuous | Historical lookback ($T_{-90} 	o T_0$) | None (0 fill if empty) | Total search result impressions over trailing 90 days. |
| `clicks_90d` | Continuous | Historical lookback ($T_{-90} 	o T_0$) | None (0 fill if empty) | Total organic clicks delivered over trailing 90 days. |
| `avg_position` | Continuous | Historical lookback ($T_{-90} 	o T_0$) | Median fill | Average SERP ranking position across all triggered queries. |
| `ctr` | Continuous | Historical lookback ($T_{-90} 	o T_0$) | Derived: $	ext{clicks} / 	ext{impr}$ | Empirical click-through rate over trailing window. |
| `days_since_last_update` | Integer | Retrospective metadata ($T_0$) | Median fill | Days elapsed since editorial modified content body. |
| `content_age_days` | Integer | Retrospective metadata ($T_0$) | Median fill | Total lifecycle days since initial URL indexing. |
| `word_count` | Integer | Retrospective metadata ($T_0$) | Median fill | Document text length indicating content depth. |
| `is_high_exposure` | Binary (0/1) | Computed at $T_0$ | Deterministic | Indicator that URL receives $\ge 500$ impressions. |
| `is_stale` | Binary (0/1) | Computed at $T_0$ | Deterministic | Indicator that URL has not been updated for $\ge 180$ days. |
| `is_striking_distance` | Binary (0/1) | Computed at $T_0$ | Deterministic | Indicator that URL ranks on Page 2 (pos 11–20). |


In [2]:
feature_summary = []
for col in X.columns:
    feature_summary.append({
        "Feature": col,
        "Dtype": str(X[col].dtype),
        "Min": round(float(X[col].min()), 3),
        "Median": round(float(X[col].median()), 3),
        "Max": round(float(X[col].max()), 3),
        "Available_At_T0": "YES (Pre-prediction)"
    })

feature_summary_df = pd.DataFrame(feature_summary)
print("Feature Vector Integrity Table:")
display(feature_summary_df)


Feature Vector Integrity Table:


,Feature,Dtype,Min,Median,Max,Available_At_T0
0,impressions_90d,int64,1.0,731.000,517715.00,YES (Pre-prediction)
1,clicks_90d,int64,0.0,1.000,4178.00,YES (Pre-prediction)
2,avg_position,float64,0.0,10.800,245.00,YES (Pre-prediction)
3,ctr,float64,0.0,0.070,100.00,YES (Pre-prediction)
4,days_since_last_update,int64,1.0,20.000,373.00,YES (Pre-prediction)
5,content_age_days,int64,90.0,236.000,564.00,YES (Pre-prediction)
6,word_count,float64,8.0,2877.000,9546.00,YES (Pre-prediction)
7,engagement_rate,float64,0.0,0.000,100.00,YES (Pre-prediction)
8,scroll_rate,float64,0.0,5.000,300.00,YES (Pre-prediction)
9,is_high_exposure,int64,0.0,1.000,1.00,YES (Pre-prediction)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

An adversarial leakage audit checks for:
1. **Target Leakage / Correlation Sniffing:** Any feature with $|r| > 0.85$ against the target label $Y$ is an immediate leak suspect.
2. **Temporal Inversion:** Verifying that no future-window performance metrics (e.g. forward 90d clicks/impressions) enter the feature matrix.
3. **Product Rule Circularity:** Ensuring internal app rule outputs (e.g. `health_score`, `priority_score`, `action_type`) are omitted.


In [3]:
# Compute Pearson correlation with the target label
correlations = X.apply(lambda col: np.corrcoef(col, y)[0, 1])
corr_df = pd.DataFrame({
    'Feature': correlations.index,
    'Target_Correlation': correlations.values,
    'Abs_Correlation': np.abs(correlations.values)
}).sort_values('Abs_Correlation', ascending=False)

print("Correlation Audit Against Binary Decline Label (y):")
display(corr_df)

# Adversarial check: fail if any feature exhibits correlation > 0.80
leaked_features = corr_df[corr_df['Abs_Correlation'] > 0.80]['Feature'].tolist()
if leaked_features:
    print(f"LEAKAGE DETECTED: {leaked_features}")
else:
    print("\nLEAKAGE AUDIT PASSED: All feature correlations are well below the 0.80 suspicious threshold.")
    print(f"Max absolute correlation: {corr_df['Abs_Correlation'].max():.4f} ({corr_df.iloc[0]['Feature']})")


Correlation Audit Against Binary Decline Label (y):


,Feature,Target_Correlation,Abs_Correlation
9,is_high_exposure,0.156051,0.156051
5,content_age_days,-0.145505,0.145505
6,word_count,0.100242,0.100242
4,days_since_last_update,0.095094,0.095094
11,is_striking_distance,0.077047,0.077047
3,ctr,-0.066179,0.066179
12,is_page_one,-0.047404,0.047404
2,avg_position,-0.037477,0.037477
8,scroll_rate,-0.020303,0.020303
13,clicks_per_age_day,-0.018073,0.018073



LEAKAGE AUDIT PASSED: All feature correlations are well below the 0.80 suspicious threshold.
Max absolute correlation: 0.1561 (is_high_exposure)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded Field | Reason for Exclusion |
|---|---|
| `content_id` | Unique synthetic identifier. Excluded to prevent arbitrary key memorization. |
| `client_id` | Tenant grouping key. Excluded from feature space to force models to generalize across tenants rather than memorizing domain authority. |
| `trend_pct` / `trend_direction` | Label-derived future trajectory features. Injecting these constitutes immediate forward-looking data leakage. |
| `impressions_last_30d` / `clicks_last_30d` | Sub-window metrics overlapping with trajectory targets; excluded to prevent temporal confusion. |
| `provider_used` / `model_used` | CMS metadata with high sparsity and zero causal connection to search engine algorithmic evaluation. |
| `cpc` / `competition` | Paid search auction metrics that introduce noise into organic informational search queries. |


In [4]:
excluded_table = pd.DataFrame([
    {"Field": "content_id", "Exclusion Reason": "Unique entity key; prevents memorization of specific URLs."},
    {"Field": "client_id", "Exclusion Reason": "Used strictly for GroupShuffleSplit partitioning; prevents tenant domain memorization."},
    {"Field": "trend_pct", "Exclusion Reason": "Forward-window target signal; direct label leakage."},
    {"Field": "trend_direction", "Exclusion Reason": "Direct target classification label; direct target leakage."},
    {"Field": "impressions_last_30d", "Exclusion Reason": "Window overlap risk with target evaluation interval."},
    {"Field": "cpc / competition", "Exclusion Reason": "Paid search advertiser auction signals; irrelevant to organic content quality."}
])
print("Excluded Fields Registry:")
display(excluded_table)


Excluded Fields Registry:


,Field,Exclusion Reason
0,content_id,Unique entity key; prevents memorization of sp...
1,client_id,Used strictly for GroupShuffleSplit partitioni...
2,trend_pct,Forward-window target signal; direct label lea...
3,trend_direction,Direct target classification label; direct tar...
4,impressions_last_30d,Window overlap risk with target evaluation int...
5,cpc / competition,Paid search advertiser auction signals; irrele...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
